# 膜電位量化:$(b, f_a, f_V, r)$ 掃描

軟體訓練端的膜電位遞迴(浮點,未量化):

$$V_k = a_k V_{k-1} + b_k, \qquad a_k=(1-1/\tau)^{\Delta t_k}$$

權重量化定案後 $b_k = s_c q_k$($q_k$ 整數碼、$s_c$ 是輸出 channel 的量化步長)。
定義 $\tilde V_k := V_k/s_c$、$\tilde v_{th} := v_{th}/s_c$,遞迴變成:

$$\tilde V_k = r\big(a_k\tilde V_{k-1}\big) + q_k, \qquad \text{fire if } \tilde V_k \ge \tilde v_{th}$$

**這份 notebook 模擬的是這條遞迴的整數版本,不是浮點近似**(見
docs/問題紀錄.md 第十七~十九節):$\tilde V_k$、$q_k$、查表出來的 $a_k$
全部是乾淨的整數,`salt_core` 內部從 $s_c$ 換算成整數只做一次(建門檻的
時候),遞迴迴圈裡完全不出現 $s_c$、不做浮點除法。$r(\cdot)$ 是捨入到 $f_V$
個小數位元(整數右移,`round` 或 `truncate`,逢五進一不是逢五取偶);$a_k$
本身也是整數查表(`build_decay_table_int`),索引直接用佇列建構時就算好的
真正整數 $\Delta t$,不是反推 $\log(a)/\log(1-1/\tau)$。

$\tilde V$ 存在 $i_V+f_V$ 位元的暫存器裡,超出範圍**真的會繞回去**(兩補數
wrap-around,不是飽和),`salt_core` 會在每一步偵測有沒有發生、回報給這份
notebook(見第 5 步)。

$(b, f_a, f_V, r)$ 掃描分四步:

1. 量 $V$(不依賴 $b$,只跑一次):$V_{\max}^{(l,c)}=\max_k V_k$、$V_{\min}^{(l,c)}=\min_k V_k$。
2. 對每個候選 $b$:算 $s_c$、$T_c=s_c(2^{b-1}-1)$,代入
   $i_V^{(l,c)}(b) = \lceil \log_2(M^{(l,c)} \cdot \frac{2^{b-1}-1}{T_c}) \rceil + 1$
   (其中 $M^{(l,c)}=\max(V_{\max}^{(l,c)}, |V_{\min}^{(l,c)}|)$)。
3. 對每個候選 $(b,r)$:在 $(f_a,f_V)$ 網格上跑上面那條整數遞迴,量準確率。
4. 從對照表選一組,拿量化模擬**實際跑出來**的溢位旗標確認沒有真的撞到 $i_V$。

量化模擬的核心邏輯(整數遞迴本身、$a_k$ 查表、$i_V$ 公式、溢位偵測)都在
`salt_core` 且有單元測試覆蓋,這裡的程式碼只是把公式接起來、跑迴圈、畫圖。

**執行時間提醒**:第 3 步的遞迴強制逐事件處理(chunk_size=1)——捨入 $r(\cdot)$ 要
「每筆事件更新後立刻捨入」才對得上硬體,合併處理多筆事件會把捨入時機算錯,所以
沒辦法批次化,是逐樣本 Python 迴圈,網格掃描組合數 × 樣本數直接疊乘。下面的預設
參數是一個可以先跑過一次的小網格,看過大概的執行時間之後再自己放大
`*_SWEEP`/`*_N_SAMPLES`。

**這份 notebook 不處理的事**:溢位政策已經定案是「直接繞回去」(不是飽和),
`salt_core` 的整數遞迴照這個政策算、也會回報有沒有真的溢位過——但選錯 $i_V$
讓它真的溢位時該怎麼辦(擴大 $i_V$、還是接受這個 $(b,f_a,f_V)$ 組合不能用),
是看完第 5 步報告之後的人為判斷,不是迴圈裡的自動修正。


In [ ]:
%matplotlib inline
import itertools
import os
import time

os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")

import matplotlib
matplotlib.rcParams["font.sans-serif"] = ["WenQuanYi Zen Hei", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False
import matplotlib.pyplot as plt
import numpy as np
import jax.numpy as jnp

from data.src.nmnist import NMNISTDataset
from example.models.conv_net import build_decoder
from example.paths import DATASET_ROOT, EXPERIMENTS_DIR
from example.replay_epoch import load_epoch_weights
from example.utils import load_run_record
from salt_core.layers import (ConvLayer, raw_events_to_stream, run_network,
                              run_network_quantized, run_network_quantized_traced,
                              run_network_traced)
from salt_core.quantize import (build_decay_table_int, iv_from_measurement, iv_layer,
                                quantize_to_int, v_th_to_int, waste)
from viz.channel_grid import unflatten_channels


## 參數

改這裡,不改函式庫。`BITS_SWEEP` 沿用 `weight_quantization_ptq.ipynb` 的候選;
`F_A_SWEEP`/`F_V_SWEEP`/`ROUND_MODES` 是這份 notebook 新掃的三個自由度。

In [ ]:
RUN_NAME = "conv_compressed_compressed_scale_10k_20260919_050446"
EXP_DIR = EXPERIMENTS_DIR / RUN_NAME
MEASURE_EPOCH = 59          # 要有對應的 weights/epoch_XXX.npz
MEASURE_N_SAMPLES = 50      # 量 V(第 1 步)、驗證溢位(第 4 步)共用,不依賴 b

BITS_SWEEP = [8, 4]              # 權重量化候選 b
F_A_SWEEP = [6, 10]              # a_k 查表小數位元數
F_V_SWEEP = [6, 10, 14]          # V 的小數位元數
ROUND_MODES = ["round", "truncate"]
ACC_N_SAMPLES = 30          # (b,f_a,f_V,r) 掃描算準確率用的樣本數(逐樣本跑,見上面執行時間提醒)
ACC_SPLIT = "val"

## 載入 checkpoint、val split

`load_epoch_weights`(`example/replay_epoch.py`)已經把每一層的 `chunk_size` 強制成 1,
跟 `forward_quantized` 要求的一致(理由見該函式 docstring:捨入要逐事件生效,
`chunk_size>1` 會把好幾筆事件合成再捨一次,答案會算錯),直接沿用不用再改一次。

In [ ]:
run_record = load_run_record(str(EXP_DIR))
data_cfg = run_record["config"]["data"]
model_cfg = run_record["config"]["model"]
decoder = build_decoder(model_cfg, None)  # membrane_regression 的 decode() 不需要 layers 資訊

dataset = NMNISTDataset(DATASET_ROOT, max_events=int(data_cfg["max_events"]))
val_split = dataset.build_split(seed=int(data_cfg["seed_val"]), n_samples=int(data_cfg["val_size"]),
                                which=ACC_SPLIT)

layers, float_params = load_epoch_weights(str(EXP_DIR), MEASURE_EPOCH)
first_layer = layers[0]
for l in layers:
    print(f"{l.name}: {type(l).__name__}  tau={l.tau}  v_th={l.v_th}  chunk_size={l.chunk_size}")

## 第 1 步:量 $V$(不依賴 $b$,只跑一次)

用原始、未量化的浮點權重(遞迴式 $V_k=a_kV_{k-1}+b_k$,完全沒有量化)跑過整個
資料集,對每層每個 channel 記錄:

$$V_{\max}^{(l,c)} = \max_k V_k, \qquad V_{\min}^{(l,c)} = \min_k V_k$$

這步不依賴 $b$(權重量化候選),只需要跑一次,之後每個候選 $b$ 重複使用同一份
測量結果。conv 層用 `unflatten_channels` 把攤平的神經元陣列還原成 $(oc,h,w)$
再逐 channel 取極值;`out`(FC)層每個神經元本身就是一個 channel,不用還原。

In [ ]:
def measure_v_ranges(layers, params, split, n_samples):
    """回傳 (v_max, v_min):兩個 dict,layer.name -> 逐 channel 的極值陣列。"""
    first = layers[0]
    v_max = {l.name: None for l in layers}
    v_min = {l.name: None for l in layers}
    for i in range(n_samples):
        sample = (split.event_times[i], split.x[i], split.y[i], split.c[i],
                 split.n_real_events[i])
        in_stream = raw_events_to_stream(*sample, h_in=first.h_in, w_in=first.w_in)
        traces = run_network_traced(layers, in_stream, params)
        for layer, trace in zip(layers, traces):
            vs = np.asarray(trace.v_steps)
            per_neuron_max = np.nanmax(vs, axis=1)
            per_neuron_min = np.nanmin(vs, axis=1)
            if isinstance(layer, ConvLayer):
                per_neuron_max = unflatten_channels(
                    per_neuron_max, layer.oc, layer.h_out, layer.w_out).max(axis=(1, 2))
                per_neuron_min = unflatten_channels(
                    per_neuron_min, layer.oc, layer.h_out, layer.w_out).min(axis=(1, 2))
            if v_max[layer.name] is None:
                v_max[layer.name], v_min[layer.name] = per_neuron_max, per_neuron_min
            else:
                v_max[layer.name] = np.maximum(v_max[layer.name], per_neuron_max)
                v_min[layer.name] = np.minimum(v_min[layer.name], per_neuron_min)
        if (i + 1) % 10 == 0:
            print(f"{i + 1}/{n_samples}")
    return v_max, v_min


v_max, v_min = measure_v_ranges(layers, float_params, val_split, MEASURE_N_SAMPLES)
M = {l.name: np.maximum(v_max[l.name], np.abs(v_min[l.name])) for l in layers}
for l in layers:
    print(f"{l.name}: M(單邊最大量值)逐 channel = {np.round(M[l.name], 2)}")

In [ ]:
fig, axes = plt.subplots(1, len(layers), figsize=(5 * len(layers), 3.5))
for ax, layer in zip(axes, layers):
    n = len(M[layer.name])
    ax.bar(np.arange(n) - 0.2, v_max[layer.name], width=0.4, label="V_max")
    ax.bar(np.arange(n) + 0.2, v_min[layer.name], width=0.4, label="V_min")
    ax.axhline(0, color="gray", linewidth=0.8)
    ax.set_title(f"{layer.name}(逐 channel V 分布,n={MEASURE_N_SAMPLES} 樣本)")
    ax.set_xlabel("channel" if isinstance(layer, ConvLayer) else "neuron")
    ax.legend()
fig.suptitle(f"{RUN_NAME}  epoch={MEASURE_EPOCH}(原始浮點權重,未量化)")
fig.tight_layout()

## 第 2 步:對每個候選 $b$,算 $T_c(b)/s_c$、$i_V(b)$

`quantize_to_int` 回傳 `(q, scale)`:`scale` 就是 $s_c$(每個 channel 的量化步長),
`q` 是整數權重碼(這一步只用 `scale` 算 $i_V$,`q` 到第 3 步才真的餵進遞迴)。乘回
$\text{levels}=2^{b-1}-1$ 換回原本的 clip 門檻 $T_c = s_c \cdot (2^{b-1}-1)$。
逐 channel 的 $i_V(b)$(signed 定點格式,含 1 個符號位,$M^{(l,c)}$ 用第 1 步量到的
$\max(V_{\max}^{(l,c)},|V_{\min}^{(l,c)}|)$):

$$i_V^{(l,c)}(b) = \left\lceil \log_2\!\left(M^{(l,c)} \cdot \frac{2^{b-1}-1}{T_c}\right) \right\rceil + 1$$

逐 layer 共用一個暫存器寬度就取這層最吃緊的 channel:$i_V^{(l)}(b)=\max_c i_V^{(l,c)}(b)$。
兩者的差距是每個 channel「被迫多付出的位元數」:

$$\text{waste}^{(l,c)}(b) = i_V^{(l)}(b) - i_V^{(l,c)}(b) \ge 0$$

waste 普遍接近 0 代表逐 layer 共用幾乎不浪費;少數 channel 遠大於其他 channel,才
值得逐 channel 分開存暫存器。**第 3 步實際餵進遞迴的是 $i_V^{(l)}(b)$(逐 layer
共用值),不是逐 channel 的細緻版本**——硬體暫存器寬度本來就是整層共用一個,
逐 channel 的版本只是拿來看 waste 大不大。


In [ ]:
def quantize_weights(layers, params, bits):
    """回傳 (q_params, scale_per_channel_list),兩者都對齊 layers。q_params
    是整數權重碼本身(quantize.quantize_to_int 的 q),不是乘回 scale 的
    w_hat——第 3 步的整數遞迴要直接吃這個。"""
    q_params, scales = [], []
    for w in params:
        q, scale = quantize_to_int(w, bits=bits, axis=0)
        q_params.append(q.astype(jnp.int32))
        scales.append(jnp.asarray(scale).reshape(-1))
    return q_params, scales


iv_by_b = {}     # b -> {layer.name: [逐 channel i_V]}
waste_by_b = {}  # b -> {layer.name: [逐 channel waste]}
for b in BITS_SWEEP:
    _q, scales = quantize_weights(layers, float_params, b)
    levels = 2 ** (b - 1) - 1
    iv_per_layer = {}
    for layer, scale in zip(layers, scales):
        T_c = np.asarray(scale) * levels
        iv_c = [iv_from_measurement(float(M[layer.name][c]), float(T_c[c]), b)
               for c in range(len(M[layer.name]))]
        iv_per_layer[layer.name] = iv_c
    iv_by_b[b] = iv_per_layer
    waste_by_b[b] = {name: waste(iv_layer(iv_c), iv_c) for name, iv_c in iv_per_layer.items()}
    print(f"b={b}:")
    for layer in layers:
        print(f"  {layer.name}: i_V 逐 channel={iv_per_layer[layer.name]}  "
             f"i_V 逐 layer={iv_layer(iv_per_layer[layer.name])}  "
             f"waste={waste_by_b[b][layer.name]}")


In [ ]:
fig, axes = plt.subplots(1, len(layers), figsize=(5 * len(layers), 3.5))
for ax, layer in zip(axes, layers):
    for b in BITS_SWEEP:
        ax.plot(waste_by_b[b][layer.name], marker="o", label=f"b={b}")
    ax.set_title(f"{layer.name}  waste(逐 channel 被迫多付出的位元數)")
    ax.set_xlabel("channel/neuron index")
    ax.set_ylabel("waste (bits)")
    ax.legend()
fig.suptitle("逐 layer 共用 vs 逐 channel:waste 越接近 0 代表共用幾乎不浪費")
fig.tight_layout()

## 第 3 步:量化模擬 forward,算準確率

實際模擬的整數遞迴,對每一筆事件(逐事件精確,`chunk_size=1`,見
`salt_core.core.process_event_int`):

$$\tilde V_k = r\big(a_k^{q}\,\tilde V_{k-1}\big) + q_k, \qquad \text{fire if } \tilde V_k \ge \tilde v_{th}^{\,q}$$

全程只有整數運算,`salt_core` 需要的量:

- `q`:整數權重碼(`quantize_to_int` 的 `q`,第 2 步只用了 `scale`,這裡才真的
  用 `q`)。
- $a_k^{q}$:`build_decay_table_int(f_a, tau)` 查出來的整數衰減碼,搭配佇列
  建構順便算出來的真正整數 $\Delta t$(不是反推)。
- $\tilde v_{th}^{\,q}$:`v_th_to_int(v_th, s_c, f_V)` 算好的整數門檻,只除一次
  $s_c$,不在迴圈裡出現。
- $i_V$:第 2 步已經算好的 $i_V^{(l)}(b)$(逐 layer 共用值),直接重用,這裡
  不重算。

conv 層的 $s_c$ 是逐 channel 的,要展開成逐神經元(`jnp.repeat` 展開
$h_{out}\times w_{out}$ 次,配 `unflatten_channels` 的攤平慣例)才能算
$\tilde v_{th}^{\,q}$;FC 層每個神經元本身就是一個 channel,不用展開。
`decoder.decode` 讀 `v_final` 當分數(`LayerForwardResultInt` 也有這個欄位)。


In [ ]:
def build_quant_params(layers, float_params, b, f_a, f_v):
    """算好單一 (b,f_a,f_v) 組合下,run_network_quantized 要的逐層參數:
    (q_params, decay_tables_int, v_th_int, f_a_list, f_V_list, i_V_list)。
    i_V 不在這裡重算,直接重用第 2 步的 iv_by_b[b](逐 layer 共用值)。"""
    q_params, decay_tables_int, v_th_int_list, i_V_list = [], [], [], []
    for layer, w in zip(layers, float_params):
        q, scale = quantize_to_int(w, bits=b, axis=0)
        scale_per_channel = jnp.asarray(scale).reshape(-1)
        q_params.append(q.astype(jnp.int32))
        decay_tables_int.append(build_decay_table_int(f_a, layer.tau))

        if isinstance(layer, ConvLayer):
            # 逐 channel 的 s_c 展開成逐神經元(channel-major,對齊
            # unflatten_channels/連接層的攤平慣例:同一個 channel 的
            # h_out*w_out 個神經元共用同一個 s_c)。
            scale_per_neuron = jnp.repeat(scale_per_channel, layer.h_out * layer.w_out)
        else:
            scale_per_neuron = scale_per_channel  # FC 每個神經元本身就是一個 channel

        v_th_int_list.append(v_th_to_int(layer.v_th, scale_per_neuron, f_v))
        i_V_list.append(iv_layer(iv_by_b[b][layer.name]))

    n = len(layers)
    return q_params, decay_tables_int, v_th_int_list, [f_a] * n, [f_v] * n, i_V_list


def quantized_accuracy(layers, float_params, split, n_samples, b, f_a, f_v, round_mode):
    q_params, decay_tables_int, v_th_int_list, f_a_list, f_V_list, i_V_list = build_quant_params(
        layers, float_params, b, f_a, f_v)
    first = layers[0]
    correct = 0
    for i in range(n_samples):
        sample = (split.event_times[i], split.x[i], split.y[i], split.c[i],
                 split.n_real_events[i])
        in_stream = raw_events_to_stream(*sample, h_in=first.h_in, w_in=first.w_in)
        result = run_network_quantized(
            layers, in_stream, q_params, decay_tables_int=decay_tables_int,
            v_th_int=v_th_int_list, f_a=f_a_list, f_V=f_V_list, i_V=i_V_list,
            round_mode=round_mode)
        scores, _ = decoder.decode(result)
        correct += int(jnp.argmax(scores) == split.labels[i])
    return correct / n_samples


def unquantized_accuracy(layers, params, split, n_samples):
    """baseline:同一批樣本、同一套已經強制 chunk_size=1 的 layers,完全不量化。"""
    first = layers[0]
    correct = 0
    for i in range(n_samples):
        sample = (split.event_times[i], split.x[i], split.y[i], split.c[i],
                 split.n_real_events[i])
        in_stream = raw_events_to_stream(*sample, h_in=first.h_in, w_in=first.w_in)
        result, _diags = run_network(layers, in_stream, params)
        scores, _ = decoder.decode(result)
        correct += int(jnp.argmax(scores) == split.labels[i])
    return correct / n_samples


baseline_accuracy = unquantized_accuracy(layers, float_params, val_split, ACC_N_SAMPLES)
print(f"baseline(未量化,chunk_size=1 精確重跑,n={ACC_N_SAMPLES}): "
     f"accuracy={baseline_accuracy:.4f}")


## 第 4 步:$(b,r)$ 每組跑一次 $(f_a,f_V)$ 網格

四個參數分三類,決定了迴圈怎麼排:$b$、$r$ 是列舉的候選值(外層迴圈);$i_V$ 第 2 步
已經用公式算完、不用搜索;$(f_a,f_V)$ 兩個都要靠實測準確率決定,是同一個搜索空間的
兩個維度,所以放在內層網格一起掃,不是先固定一個再掃另一個。

In [ ]:
sweep_results = []
t0 = time.time()
for b, r in itertools.product(BITS_SWEEP, ROUND_MODES):
    for f_a, f_v in itertools.product(F_A_SWEEP, F_V_SWEEP):
        acc = quantized_accuracy(layers, float_params, val_split, ACC_N_SAMPLES, b, f_a, f_v, r)
        sweep_results.append(dict(b=b, round_mode=r, f_a=f_a, f_v=f_v, accuracy=acc))
        print(f"b={b} r={r:9s} f_a={f_a:2d} f_v={f_v:2d}: accuracy={acc:.4f}")
print(f"總耗時 {time.time() - t0:.0f}s,共 {len(sweep_results)} 組")

In [ ]:
combos = sorted(set((x["b"], x["round_mode"]) for x in sweep_results))
fig, axes = plt.subplots(1, len(combos), figsize=(4.5 * len(combos), 4), squeeze=False)
axes = axes[0]
for ax, (b, r) in zip(axes, combos):
    sub = [x for x in sweep_results if x["b"] == b and x["round_mode"] == r]
    grid = np.full((len(F_A_SWEEP), len(F_V_SWEEP)), np.nan)
    for x in sub:
        grid[F_A_SWEEP.index(x["f_a"]), F_V_SWEEP.index(x["f_v"])] = x["accuracy"]
    im = ax.imshow(grid, vmin=0, vmax=1, cmap="viridis")
    ax.set_xticks(range(len(F_V_SWEEP))); ax.set_xticklabels(F_V_SWEEP)
    ax.set_yticks(range(len(F_A_SWEEP))); ax.set_yticklabels(F_A_SWEEP)
    ax.set_xlabel("f_V"); ax.set_ylabel("f_a")
    ax.set_title(f"b={b}, r={r}")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.suptitle(f"量化模擬準確率對照表(baseline={baseline_accuracy:.4f}, n={ACC_N_SAMPLES})")
fig.tight_layout()

## 第 5 步:選一組,驗證沒有溢位

從對照表挑一組(下面預設挑準確率最高的,實際要不要選這組、要不要在準確率跟
硬體成本之間找別的平衡點,是看完對照表之後的判斷,不是這裡自動決定)。

溢位政策已經定案是「直接繞回去」(兩補數 wrap-around,不是飽和,見
docs/問題紀錄.md 第十七節),`salt_core` 的整數遞迴(`core.process_event_int`)
每一步都會實際偵測「這一步寫回暫存器有沒有撞到 $i_V+f_V$ 位元的寬度」,不是
事後拿測量值去跟公式算出來的理論上界比對——這裡直接讀這個真溢位旗標,對
`run_network_quantized_traced` 逐層、逐步疊出來的 `overflowed` 陣列取
`any()`,只要任何一個樣本、任何一步、任何一個神經元撞到過,就代表這組
$(b,f_a,f_V)$ 選的 $i_V$ 不夠寬。

順便換算回原始 $V$ 尺度($V=\tilde V\cdot s_c$)畫出量化模擬實際跑出來的
$M^{(l,c)}$,跟第 1 步(未量化浮點模型)量到的範圍對照,當作理解用的輔助
資訊——真正的判斷依據是上面那個真溢位旗標,不是這個對照本身。


In [ ]:
best = max(sweep_results, key=lambda x: x["accuracy"])
print(f"挑選:b={best['b']} f_a={best['f_a']} f_v={best['f_v']} r={best['round_mode']}  "
     f"accuracy={best['accuracy']:.4f}(baseline={baseline_accuracy:.4f})")

q_params, decay_tables_int, v_th_int_list, f_a_list, f_V_list, i_V_list = build_quant_params(
    layers, float_params, best["b"], best["f_a"], best["f_v"])
_, scales = quantize_weights(layers, float_params, best["b"])

quant_v_max = {l.name: None for l in layers}   # 整數暫存器的原始值(還沒換算回 V 尺度)
quant_v_min = {l.name: None for l in layers}
overflow_any = {l.name: None for l in layers}  # 逐 channel/neuron,這個 layer 有沒有真的撞過溢位
for i in range(MEASURE_N_SAMPLES):
    sample = (val_split.event_times[i], val_split.x[i], val_split.y[i], val_split.c[i],
             val_split.n_real_events[i])
    in_stream = raw_events_to_stream(*sample, h_in=first_layer.h_in, w_in=first_layer.w_in)
    traces = run_network_quantized_traced(
        layers, in_stream, q_params, decay_tables_int=decay_tables_int,
        v_th_int=v_th_int_list, f_a=f_a_list, f_V=f_V_list, i_V=i_V_list,
        round_mode=best["round_mode"])
    for layer, (result, v_steps) in zip(layers, traces):
        vs = np.asarray(v_steps)
        pmax = np.nanmax(vs, axis=1)
        pmin = np.nanmin(vs, axis=1)
        ov = np.asarray(result.overflowed).any(axis=1)  # 這個樣本,每個神經元有沒有撞過溢位
        if isinstance(layer, ConvLayer):
            pmax = unflatten_channels(pmax, layer.oc, layer.h_out, layer.w_out).max(axis=(1, 2))
            pmin = unflatten_channels(pmin, layer.oc, layer.h_out, layer.w_out).min(axis=(1, 2))
            ov = unflatten_channels(ov.astype(np.int32), layer.oc, layer.h_out,
                                    layer.w_out).any(axis=(1, 2))
        if quant_v_max[layer.name] is None:
            quant_v_max[layer.name], quant_v_min[layer.name] = pmax, pmin
            overflow_any[layer.name] = ov
        else:
            quant_v_max[layer.name] = np.maximum(quant_v_max[layer.name], pmax)
            quant_v_min[layer.name] = np.minimum(quant_v_min[layer.name], pmin)
            overflow_any[layer.name] = overflow_any[layer.name] | ov

for layer, scale, iv_layer_val, f_v_used in zip(layers, scales, i_V_list, f_V_list):
    scale_per_channel = np.asarray(scale)
    # v_steps 是整數暫存器的原始值(Ṽ*2^f_V 的整數計數),換算回原始 V 尺度
    # 才能跟第 1 步(未量化浮點模型)量到的範圍對照。
    quant_M_int = np.maximum(quant_v_max[layer.name], np.abs(quant_v_min[layer.name]))
    quant_M = quant_M_int / (2.0 ** f_v_used) * scale_per_channel
    representable = 2.0 ** (iv_layer_val - 1) * scale_per_channel  # 這層實際用的 i_V(逐 layer 共用)
    overflow = overflow_any[layer.name]
    print(f"{layer.name}: 量化模擬實測 M(換算回原始 V 尺度)={np.round(quant_M, 2)}")
    print(f"        i_V={iv_layer_val} 隱含上界={np.round(representable, 2)}")
    print(f"        整數硬體真溢位偵測(逐 channel/neuron)={overflow.tolist()}")
    if overflow.any():
        print("        !! 有 channel 真的溢位過(整數暫存器繞回去了),"
             "這組 (b,f_a,f_V,r) 的 i_V 需要重新算/加安全邊界")
